
<img src="https://i.stack.imgur.com/aTDpS.png">
<img src="http://colah.github.io/posts/2015-08-Understanding-LSTMs/img/LSTM3-focus-f.png">
<img src="http://colah.github.io/posts/2015-08-Understanding-LSTMs/img/LSTM3-focus-i.png">
<img src="http://colah.github.io/posts/2015-08-Understanding-LSTMs/img/LSTM3-focus-C.png">
<img src="http://colah.github.io/posts/2015-08-Understanding-LSTMs/img/LSTM3-focus-o.png">

<img src="https://leonardoaraujosantos.gitbooks.io/artificial-inteligence/content/assets/LSTMBlockDiagram.png">


In [1]:
'''Train a Bidirectional LSTM on the IMDB sentiment classification task.
Output after 4 epochs on CPU: ~0.8146
Time per epoch on CPU (Core i7): ~150s.
'''

from __future__ import print_function
import numpy as np
np.random.seed(1337)  # for reproducibility

from keras.preprocessing import sequence
from keras.models import Model
from keras.layers import Dense, Dropout, Embedding, LSTM, Input, merge
from keras.datasets import imdb
# Dataset of 25,000 movies reviews from IMDB, labeled by sentiment (positive/negative).
# Reviews have been preprocessed, and each review is encoded as a sequence of word indexes (integers).



Using TensorFlow backend.


In [2]:

max_features  = 20000
maxlen        = 100  # cut texts after this number of words (among top max_features most common words)
batch_size    = 32
emb_dim       = 50

print('Loading data...')
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)
print(len(x_train), 'train sequences')
print(len(x_test), 'test sequences')


Loading data...
25000 train sequences
25000 test sequences


In [3]:

print("Pad sequences (samples x time)")
x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)
print('X_train shape:', x_train.shape)
print('X_test shape:', x_test.shape)
y_train = np.array(y_train)
y_test = np.array(y_test)


Pad sequences (samples x time)
X_train shape: (25000, 100)
X_test shape: (25000, 100)



<img src="https://i.stack.imgur.com/aTDpS.png">


<img src="https://cdn-images-1.medium.com/max/1156/1*laH0_xXEkFE0lKJu54gkFQ.png">



In [4]:

# this is the placeholder tensor for the input sequences
sequence = Input(shape=(maxlen,), dtype='int32')
# this embedding layer will transform the sequences of integers
# into vectors of size 128
emb_layer = Embedding(max_features, emb_dim, input_length=maxlen)
embedded = emb_layer(sequence)
# apply forwards LSTM
forwards = LSTM(64)(embedded)
# apply backwards LSTM
backwards = LSTM(64, go_backwards=True)(embedded)
# concatenate the outputs of the 2 LSTMs
merged = merge([forwards, backwards], mode='concat', concat_axis=-1)
# after_dp = Dropout(0.5)(merged)
# output = Dense(1, activation='sigmoid')(after_dp)
output = Dense(1, activation='sigmoid')(merged)
model = Model(input=sequence, output=output)


/home/user/virtual_environments/my_project/venv/lib/python2.7/site-packages/ipykernel/__main__.py:13: UserWarning: The `merge` function is deprecated and will be removed after 08/2017. Use instead layers from `keras.layers.merge`, e.g. `add`, `concatenate`, etc.
/home/user/virtual_environments/my_project/venv/local/lib/python2.7/site-packages/keras/legacy/layers.py:458: UserWarning: The `Merge` layer is deprecated and will be removed after 08/2017. Use instead layers from `keras.layers.merge`, e.g. `add`, `concatenate`, etc.
  name=name)
/home/user/virtual_environments/my_project/venv/lib/python2.7/site-packages/ipykernel/__main__.py:17: UserWarning: Update your `Model` call to the Keras 2 API: `Model(outputs=Tensor("de..., inputs=Tensor("in...)`


In [5]:

# try using different optimizers and different optimizer configs
model.compile('adam', 'binary_crossentropy', metrics=['accuracy'])

print('Train...')
model.fit(
    x_train[:9000],
    y_train[:9000],
    batch_size=batch_size,
    epochs=5,
    validation_data=[
        x_train[9000:10000],
        y_train[9000:10000]
    ]
)


Train...
Train on 9000 samples, validate on 1000 samples
Epoch 1/5
9000/9000 [==============================] - 219s - loss: 0.5139 - acc: 0.7384 - val_loss: 0.3890 - val_acc: 0.8160

In [6]:
score = model.evaluate(
    x_test,                  # features
    y_test,                  # labels
    batch_size=batch_size,   # batch size
    verbose=1                # the mostX_test extended verbose
)


print('\nTest categorical_crossentropy:', score[0])
print('\nTest accuracy:', score[1])



25000/25000 [==============================] - 158s     